In [61]:
import numpy as np
import pandas as pd

In [62]:
RAW_PATH = "uploads/floodsense_training_data.csv"
ELEV_PATH = "uploads/district_elevation_reference.csv"
OUT_PATH = "processed/floodsense_clean.csv"
audit = {}
def section(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


In [63]:
section("LOADING RAW DATA")
df = pd.read_csv(RAW_PATH)
elev = pd.read_csv(ELEV_PATH)
print(f"Raw shape: {df.shape}")
audit["original_rows"] = len(df)
audit["original_cols"] = df.shape[1]


LOADING RAW DATA
Raw shape: (1434, 25)


In [64]:
section("PHASE A — CLEAN VALUES")
df["date"] = pd.to_datetime(df["date"])
print(f"A1. Dates parsed. Range: {df['date'].min().date()} to {df['date'].max().date()}")


PHASE A — CLEAN VALUES
A1. Dates parsed. Range: 2022-01-01 to 2024-12-30


In [65]:
df["date"] = pd.to_datetime(df["date"])
print(f"A1. Dates parsed. Range: {df['date'].min().date()} to {df['date'].max().date()}")

A1. Dates parsed. Range: 2022-01-01 to 2024-12-30


In [66]:
sentinels = [-999, 99999]
n_sent = 0
print("A2. Replacing sentinel values with NaN:")
for col in df.select_dtypes(include=np.number).columns:
    for s in sentinels:
        mask = df[col] == s
        if mask.any():
            print(f"      {col}: {mask.sum()} instances of {s}")
            df.loc[mask, col] = np.nan
            n_sent += mask.sum()
audit["sentinels_replaced"] = int(n_sent)
print(f"      Total sentinel cells replaced: {n_sent}")

A2. Replacing sentinel values with NaN:
      elevation: 1 instances of 99999
      precipitation: 1 instances of -999
      temperature: 1 instances of -999
      Total sentinel cells replaced: 3


In [67]:
domain_rules = [
    ("humidity",      lambda x: (x < 0) | (x > 100),  "∉ [0, 100]%"),
    ("soil_moisture", lambda x: (x < 0) | (x > 1),    "∉ [0, 1] scale"),
    ("temperature",   lambda x: (x < -20) | (x > 60), "∉ [-20°C, 60°C]"),
    ("precipitation", lambda x: x < 0,                "< 0"),
    ("pressure",      lambda x: x < 80000,            "< 80,000 Pa"),
]
n_imp = 0
print("A3. Replacing domain-rule violations with NaN:")
for col, rule, desc in domain_rules:
    if col in df.columns:
        mask = rule(df[col]) & df[col].notna()
        if mask.any():
            print(f"      {col} {desc}: {mask.sum()} rows")
            df.loc[mask, col] = np.nan
            n_imp += mask.sum()
audit["impossibles_replaced"] = int(n_imp)
print(f"      Total impossible values replaced: {n_imp}")

A3. Replacing domain-rule violations with NaN:
      humidity ∉ [0, 100]%: 1 rows
      soil_moisture ∉ [0, 1] scale: 1 rows
      Total impossible values replaced: 2


In [68]:
n_inf = int(np.isinf(df.select_dtypes(include=np.number)).sum().sum())
df = df.replace([np.inf, -np.inf], np.nan)
audit["inf_replaced"] = n_inf
print(f"A4. Replaced {n_inf} infinite value(s) with NaN")

A4. Replaced 1 infinite value(s) with NaN


In [69]:
df["district"] = df["district"].astype(str).str.strip()
print("A5. Standardized 'district' column (whitespace stripped)")

A5. Standardized 'district' column (whitespace stripped)


In [70]:
section("PHASE B — DROP BAD ROWS")
PHANTOM_NAN_THRESHOLD = 10
nan_per_row = df.isna().sum(axis=1)
phantom_mask = nan_per_row > PHANTOM_NAN_THRESHOLD
print(f"B1. Phantom row detection (threshold: > {PHANTOM_NAN_THRESHOLD} NaN per row)")
print(f"      Distribution of NaN-counts per row: max={nan_per_row.max()}, "
      f"mean={nan_per_row.mean():.2f}, median={int(nan_per_row.median())}")
if phantom_mask.any():
    phantom_summary = df.loc[phantom_mask, ["date", "district"]].copy()
    phantom_summary["nan_count"] = nan_per_row[phantom_mask]
    print("      Phantom rows identified:")
    for _, row in phantom_summary.iterrows():
        print(f"        date={row['date'].date()}  district={row['district']}  "
              f"nan_count={row['nan_count']}")
df = df[~phantom_mask].copy()
audit["phantom_rows_dropped"] = int(phantom_mask.sum())
print(f"      Dropped {audit['phantom_rows_dropped']} phantom rows. Remaining: {len(df)}")


PHASE B — DROP BAD ROWS
B1. Phantom row detection (threshold: > 10 NaN per row)
      Distribution of NaN-counts per row: max=19, mean=0.18, median=0
      Phantom rows identified:
        date=2023-08-15  district=Nowshera  nan_count=19
        date=2023-09-02  district=Jacobabad  nan_count=19
      Dropped 2 phantom rows. Remaining: 1432


In [71]:
n_before = len(df)
df = df.drop_duplicates()
audit["exact_duplicates_dropped"] = n_before - len(df)
print(f"B2. Dropped {audit['exact_duplicates_dropped']} exact duplicate rows. Remaining: {len(df)}")

B2. Dropped 67 exact duplicate rows. Remaining: 1365


In [72]:
n_dup_keys = int(df.duplicated(subset=["date", "district"]).sum())
audit["date_district_duplicates_kept"] = n_dup_keys
print(f"B3. Keeping {n_dup_keys} (date, district) duplicate rows as separate observations")
print("      (Decision: same date/district with different sensor readings = independent samples)")

B3. Keeping 270 (date, district) duplicate rows as separate observations
      (Decision: same date/district with different sensor readings = independent samples)


In [73]:
section("PHASE C — DROP BAD COLUMNS")
cols_to_drop = ["ds_idx", "elevation", "latitude", "longitude"]
existing = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing)
audit["columns_dropped"] = existing
print(f"Dropped: {existing}")
print("  ds_idx                  → target leak (verified corr = -0.70)")
print("  elevation, lat, lon     → constant within each district (no signal)")


PHASE C — DROP BAD COLUMNS
Dropped: ['ds_idx', 'elevation', 'latitude', 'longitude']
  ds_idx                  → target leak (verified corr = -0.70)
  elevation, lat, lon     → constant within each district (no signal)


In [74]:
section("PHASE D — AUGMENT")

elev_subset = elev[["district", "avg_elevation_m"]].copy()
elev_subset["district"] = elev_subset["district"].astype(str).str.strip()
df = df.merge(elev_subset, on="district", how="left")
n_unmatched = int(df["avg_elevation_m"].isna().sum())
audit["elevation_unmatched"] = n_unmatched
print(f"D1. Merged avg_elevation_m from elevation reference.")
print(f"      Districts unmatched in reference: {n_unmatched} rows")


PHASE D — AUGMENT
D1. Merged avg_elevation_m from elevation reference.
      Districts unmatched in reference: 0 rows


In [75]:
df["is_missing_precipitation"] = df["precipitation"].isna().astype(int)
n_miss = int(df["is_missing_precipitation"].sum())
audit["precipitation_missing"] = n_miss
print(f"D2. Added 'is_missing_precipitation' flag. {n_miss} rows have missing precipitation.")
print("      (We do NOT impute — LightGBM handles NaN natively in numeric features.)")

D2. Added 'is_missing_precipitation' flag. 205 rows have missing precipitation.
      (We do NOT impute — LightGBM handles NaN natively in numeric features.)


In [ ]:
section("PHASE E — FINALIZE")

df = df.sort_values(["date", "district"]).reset_index(drop=True)
print("E1. Sorted by (date, district) and reset index — required for time-aware CV")

df.to_csv(OUT_PATH, index=False)
audit["final_rows"] = len(df)
audit["final_cols"] = df.shape[1]
print(f"E2. Saved cleaned data → {OUT_PATH}")
print(f"      Final shape: {df.shape}")


PHASE E — FINALIZE
E1. Sorted by (date, district) and reset index — required for time-aware CV
E2. Saved cleaned data → processed/floodsense_clean.csv
      Final shape: (1365, 23)


In [77]:
print("Keys in audit:", list(audit.keys()))

Keys in audit: ['original_rows', 'original_cols', 'sentinels_replaced', 'impossibles_replaced', 'inf_replaced', 'phantom_rows_dropped', 'exact_duplicates_dropped', 'date_district_duplicates_kept', 'columns_dropped', 'elevation_unmatched', 'precipitation_missing', 'final_rows', 'final_cols']


In [78]:
section("AUDIT SUMMARY — for pitch deck")

print(f"Original:    {audit['original_rows']:>5} rows × {audit['original_cols']} columns")
print(f"Final:       {audit['final_rows']:>5} rows × {audit['final_cols']} columns")
print()
print("Cell-level changes:")
print(f"  Sentinel cells (-999, 99999) → NaN:    {audit['sentinels_replaced']}")
print(f"  Domain-violating cells → NaN:          {audit['impossibles_replaced']}")
print(f"  Infinite values → NaN:                 {audit['inf_replaced']}")
print()
print("Row-level changes:")
print(f"  Phantom rows dropped:                  {audit['phantom_rows_dropped']}")
print(f"  Exact duplicate rows dropped:          {audit['exact_duplicates_dropped']}")
print(f"  Date-district duplicates kept:         {audit['date_district_duplicates_kept']}")
print()
print("Column-level changes:")
print(f"  Columns dropped:                       {audit['columns_dropped']}")
print(f"  Columns added:                         ['avg_elevation_m', 'is_missing_precipitation']")
print()
print("District distribution after cleaning:")
print(df["district"].value_counts().to_string().replace("\n", "\n  "))
print()
print("Class balance after cleaning:")
balance = df["flood_event"].value_counts(normalize=True).round(3).to_dict()
print(f"  flood_event=0: {balance.get(0, 0):.1%}    flood_event=1: {balance.get(1, 0):.1%}")


AUDIT SUMMARY — for pitch deck
Original:     1434 rows × 25 columns
Final:        1365 rows × 23 columns

Cell-level changes:
  Sentinel cells (-999, 99999) → NaN:    3
  Domain-violating cells → NaN:          2
  Infinite values → NaN:                 1

Row-level changes:
  Phantom rows dropped:                  2
  Exact duplicate rows dropped:          67
  Date-district duplicates kept:         270

Column-level changes:
  Columns dropped:                       ['ds_idx', 'elevation', 'latitude', 'longitude']
  Columns added:                         ['avg_elevation_m', 'is_missing_precipitation']

District distribution after cleaning:
district
  Sindh_District          456
  Balochistan_District    455
  KP_District             454

Class balance after cleaning:
  flood_event=0: 67.6%    flood_event=1: 32.4%
